#### _0 Check which of the selected settlements were actually surveyed

Author : KT Hobbs

In [ ]:
import pandas as pd 
import geopandas as gpd
from src.config import *
import shapely
import glob
import os

#### 0. Set-up

In [ ]:
data=pd.read_csv(DIR_RAW_DATA)
settlements_chosen=gpd.read_file(SAMPLE_PATH/"sample/benin/benin_sample_46_V2.1_MS_SS.geojson")

Convert to geodataframes

In [ ]:
data_gdf = gpd.GeoDataFrame(data,
            geometry=gpd.points_from_xy(data['gps-Longitude'], data['gps-Latitude']),
            crs='epsg:4326'
)

#### 1. Assign station name to survey sample

distance threshold of 1 km for snapping to nearest station

In [ ]:
# Initialize a column for station labels if it doesn't exist
if 'station_label' not in data.columns:
    data['station_label'] = None

# Convert data_gdf to projected CRS for accurate distance calculations
data_gdf_projected = data_gdf.to_crs('epsg:32632')

# Collect all station polygons for later nearest-neighbor search
all_stations = []

# Iterate through each station file
for station_file in glob.glob(STATION_PATH + "*.gpkg"):
    # Read the station polygon file
    station_gdf = gpd.read_file(station_file)
    
    # Ensure CRS matches
    if station_gdf.crs != data_gdf.crs:
        station_gdf = station_gdf.to_crs(data_gdf.crs)
    
    # Convert to projected CRS for distance calculations
    station_gdf_projected = station_gdf.to_crs('epsg:32632')
    
    # Extract station name from filename
    station_name = station_file.split('/')[-1].replace('.gpkg', '')
    
    # Add station name to the GeoDataFrame
    station_gdf_projected['station_name'] = station_name
    
    # Store for later nearest-neighbor search
    all_stations.append(station_gdf_projected)
    
    # Perform spatial join to find points within polygons
    points_in_station = gpd.sjoin(data_gdf_projected, station_gdf_projected, how='inner', predicate='within')
    
    # Update the station_label for matching points
    if len(points_in_station) > 0:
        data.loc[points_in_station.index, 'station_label'] = station_name
        print(f"Assigned {len(points_in_station)} points to station: {station_name}")

print(f"\nTotal points labeled (within polygons): {data['station_label'].notna().sum()}")
print(f"Total points unlabeled: {data['station_label'].isna().sum()}")

# Snap unmatched points to nearest station
unmatched_indices = data[data['station_label'].isna()].index
unmatched_points = data_gdf_projected.loc[unmatched_indices]

# Distance threshold in meters (1 km)
MAX_DISTANCE = 1000

if len(unmatched_points) > 0 and len(all_stations) > 0:
    print(f"\nSnapping {len(unmatched_points)} unmatched points to nearest stations (within {MAX_DISTANCE}m)...")
    
    # Combine all station polygons into one GeoDataFrame
    all_stations_gdf = pd.concat(all_stations, ignore_index=True)
    # Reset index to ensure clean integer indexing
    all_stations_gdf = all_stations_gdf.reset_index(drop=True)
    
    snapped_count = 0
    
    # For each unmatched point, find nearest station
    for idx in unmatched_indices:
        point_geom = data_gdf_projected.loc[idx, 'geometry']
        
        # Calculate distance to all stations
        all_stations_gdf['distance'] = all_stations_gdf.geometry.distance(point_geom)
        
        # Get the position of minimum distance
        min_dist_position = all_stations_gdf['distance'].argmin()
        nearest_station = all_stations_gdf.iloc[min_dist_position]
        
        # Only snap if within threshold distance
        if nearest_station['distance'] <= MAX_DISTANCE:
            data.loc[idx, 'station_label'] = nearest_station['station_name']
            snapped_count += 1
    
    print(f"Snapped {snapped_count} points to nearest stations.")
    print(f"{len(unmatched_points) - snapped_count} points were beyond {MAX_DISTANCE}m threshold.")

# Final summary
print(f"\nTotal points labeled: {data['station_label'].notna().sum()}")
print(f"Total points unlabeled: {data['station_label'].isna().sum()}")

#### Export surveys that occured outside of station boundaries

In [ ]:
data.loc[data.station_label.isna()].to_csv(DIR_CLEANED.joinpath("_0_points_outside_stations.csv"), index=False)

#### 2. Repeat for settlement names

In [ ]:
# Initialize a column for station labels if it doesn't exist
if 'settlement_label' not in data.columns:
    data['settlement_label'] = None

# Read the preselected settlement file
settlement_df = gpd.read_file(SAMPLE_PATH.joinpath("preselection", "benin", "0_benin_surveybounds_V2.1_MS_SS.geojson"))

# Read OSM buffered settlements
settlements_osm = gpd.read_file(SAMPLE_PATH.joinpath("../input/settlement_areas/benin/GRID3_BEN_Settlement_Extents_v3_0_8527819486404085423.geojson"))


In [ ]:
settlements_osm.head(2)

In [ ]:
import numpy as np

# convert joining dtypes
settlement_df['geom_id'] = settlement_df['geom_id'].astype(str)
settlements_osm['geom_id'] = settlements_osm['osm_id'].astype(str)

# join based on geomid
settlement_full = settlement_df.merge(
    settlements_osm,
    on='geom_id',
    how='right',
    suffixes=["_preselect", "_osm"]
)

# Convert to GPD
settlement_gdf = gpd.GeoDataFrame(settlement_full,
                                   geometry='geometry',
                                   crs='epsg:4326')

# Ensure CRS matches
if settlement_gdf.crs != data_gdf.crs:
    settlement_gdf = settlement_gdf.to_crs(data_gdf.crs)

# Convert to projected CRS for accurate distance and area calculations
settlement_gdf = settlement_gdf.to_crs('epsg:32632')
data_gdf_projected = data_gdf.to_crs('epsg:32632')

# Calculate area for each polygon (now in square meters)
settlement_gdf['polygon_area'] = settlement_gdf.geometry.area

# First, try to find points within polygons
points_in_settlement = gpd.sjoin(data_gdf_projected, settlement_gdf, how='inner', predicate='within')

# For points within polygons, if multiple matches exist, keep the largest polygon
if len(points_in_settlement) > 0:
    # Group by original index and keep the row with largest polygon_area
    points_in_settlement = points_in_settlement.loc[
        points_in_settlement.groupby(points_in_settlement.index)['polygon_area'].idxmax()
    ]
    
    data.loc[points_in_settlement.index, 'settlement_label'] = points_in_settlement['name'].values
    data.loc[points_in_settlement.index, 'settlement_id'] = points_in_settlement['geom_id'].values
    print(f"Assigned {len(points_in_settlement)} points to settlements (within polygons).")

# Find points that weren't matched (still unlabeled)
unmatched_indices = data[data['settlement_id'].isna()].index
unmatched_points = data_gdf_projected.loc[unmatched_indices]

if len(unmatched_points) > 0:
    print(f"\nSnapping {len(unmatched_points)} unmatched points to nearest settlements...")
    
    # For each unmatched point, find nearest settlement
    for idx in unmatched_indices:
        point_geom = data_gdf_projected.loc[idx, 'geometry']
        
        # Calculate distance to all settlements (now in meters)
        settlement_gdf['distance'] = settlement_gdf.geometry.distance(point_geom)
        
        # Find settlements within a reasonable distance (optional: set a threshold)
        # nearby_settlements = settlement_gdf[settlement_gdf['distance'] < MAX_DISTANCE]
        
        # Get the nearest settlements
        min_distance = settlement_gdf['distance'].min()
        nearest_settlements = settlement_gdf[settlement_gdf['distance'] == min_distance]
        
        # Among nearest, select the one with largest area
        if len(nearest_settlements) > 0:
            largest_nearest = nearest_settlements.loc[nearest_settlements['polygon_area'].idxmax()]
            
            data.loc[idx, 'settlement_label'] = largest_nearest['name']
            data.loc[idx, 'settlement_id'] = largest_nearest['geom_id']
    
    print(f"Snapped {len(unmatched_points)} points to nearest largest settlements.")

# Final summary
print(f"\nTotal points labeled: {data['settlement_id'].notna().sum()}")
print(f"Total points unlabeled: {data['settlement_id'].isna().sum()}")

In [ ]:
data.loc[data.settlement_id.isna()].to_csv(DIR_CLEANED.joinpath("_0_points_outside_settlements.csv"), index=False)

##### Use enumerators inputs to label settlement

In [ ]:
# 1. Normalize spelling BEFORE grouping
def clean_location_other(x):
    if isinstance(x, str):
        x_clean = x.strip().lower()
        if x_clean in ["dadin kowa", "dadin", "dadinkowa", "dadin kowa"]:
            return "Dadin Kowa"
        return x.strip()
    return x

data["location_other_clean"] = data["location_other"].map(clean_location_other)


In [ ]:
# Build dictionary for other name
name_dict_other = (
    data[['location', 'location_other_clean']]
    .dropna()
    .groupby('location')['location_other_clean']
    .apply(lambda x: sorted(set(x)))   # ensure unique + sorted
    .to_dict()
)

# 3. Special handling for -99: expand list into separate numbered keys
if -99 in name_dict_other:
    values = name_dict_other[-99]

    expanded = {
        -99 + i/10 : val   # -99.0, -99.1, -99.2...
        for i, val in enumerate(values)
    }

    # remove the original -99 entry
    del name_dict_other[-99]

    # add expanded keys
    name_dict_other.update(expanded)

In [ ]:
# Create dictionaries mapping the location number to attributes
name_dict = settlements_chosen.set_index('sample_number')['name'].to_dict()

# update name dict to include -99
name_dict.update(name_dict_other)

# create dict for ids
id_dict = settlements_chosen.set_index('sample_number')['geom_id'].to_dict()

# Map onto data
data.loc[data.settlement_label.isna(), 'settlement_label'] = data.loc[data.settlement_label.isna(), 'location'].map(lambda x: name_dict[x])
data.loc[data.settlement_id.isna(), 'settlement_id'] = data.loc[data.settlement_id.isna(), 'location'].map(lambda x: id_dict[x])

In [ ]:
# add location label to all and compare to settlement (geo join) label
data['location_label'] = data.location.map(lambda x: name_dict[x])

In [ ]:
data.loc[data.settlement_label != data.location_label]

In [ ]:
name_dict_other

In [ ]:
# are they the same?
data.groupby(['settlement_label', 'location_label'], dropna=False).count()

In [ ]:
# . sanity check
data.loc[data.settlement_id.isna()]

In [ ]:
name_dict

Add province name

In [ ]:
state_dict = {
    3:	'Maradi',
    2: 'Tahoua',
    1:'Zinder',
}

data['state']= data.province.map(state_dict)

Export

In [ ]:
data.to_csv(DIR_CLEANED.joinpath("_0_data_settlement_ids.csv"), index=False)